# 🌙 Al-Ansar Voice Cloning، سرور

**3 قدم:**
1. اوپر مینو: **Runtime ← Change runtime type ← T4 GPU**
2. نیچے سیل میں اپنا **GitHub یوزر نیم** لکھیں، پھر **▶ بٹن ایک بار** دبائیں
3. آخر میں ایک **ہرا لنک** آئے گا۔ اسے فون یا پی سی پر کھولیں، یا ایپ میں چسپاں کریں۔

> یہ سیل چلتا رہنا چاہیے۔ بند کریں گے تو ایپ کام نہیں کرے گی۔ پہلی بار ماڈل ڈاؤن لوڈ ہونے میں 10 سے 15 منٹ لگتے ہیں۔

In [ ]:
#@title ▶ سرور چلائیں
GITHUB_USER = "salmanabbasinm-hash"  #@param {type:"string"}
REPO = "AlAnsar-Voice-Cloning"        #@param {type:"string"}

import os, re, sys, time, secrets, subprocess

def sh(c): subprocess.run(c, shell=True, check=True)

try:
    import torch
    assert torch.cuda.is_available()
except Exception:
    raise RuntimeError("❌ GPU نہیں ملا۔ Runtime ← Change runtime type ← T4 GPU چنیں، پھر دوبارہ چلائیں۔")

print("⏳ فائلیں اور لائبریریاں انسٹال ہو رہی ہیں...")
sh("rm -rf /content/app && git clone --depth 1 https://github.com/%s/%s /content/app" % (GITHUB_USER, REPO))
sh("pip -q install -r /content/app/backend/requirements.txt")
sh("apt -qq install -y ffmpeg > /dev/null 2>&1")
sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared")

KEY = secrets.token_urlsafe(9)
env = dict(os.environ, ALANSAR_KEY=KEY)
log = open("/content/server.log", "w")
srv = subprocess.Popen([sys.executable, "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
                       cwd="/content/app/backend", env=env, stdout=log, stderr=subprocess.STDOUT)
time.sleep(5)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 90 and url is None:
    line = tun.stdout.readline()
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
    if m: url = m.group(0)
if not url:
    raise RuntimeError("❌ لنک نہیں بن سکا، سیل دوبارہ چلائیں")

app_link = "https://%s.github.io/%s/?server=%s&key=%s" % (GITHUB_USER, REPO, url, KEY)
from IPython.display import display, HTML
display(HTML('<div style="font-size:18px;line-height:2;direction:rtl">✅ <b>سرور چل رہا ہے</b><br>'
             '👉 <a href="%s" target="_blank" style="color:#0f766e;font-weight:bold">یہاں دبا کر ایپ کھولیں</a><br>'
             '<span style="font-size:14px">APK ایپ میں چسپاں کرنے کے لیے یہ پورا لنک کاپی کریں:</span><br>'
             '<input style="width:100%%;direction:ltr" value="%s" onclick="this.select()"></div>' % (app_link, app_link)))
print("\n⏳ ماڈل لوڈ ہو رہے ہیں۔ ایپ میں اوپر 🟢 آنے تک انتظار کریں۔ اس سیل کو چلتا رہنے دیں۔")

while True:
    time.sleep(120)
    if srv.poll() is not None:
        print("⚠️ سرور بند ہو گیا۔ آخری لاگ:")
        print(open("/content/server.log").read()[-1500:])
        break

### مسئلے اور حل
- **ایپ میں 🔴 سرور سے رابطہ نہیں:** سیل بند ہو گیا ہوگا۔ دوبارہ چلائیں اور نیا لنک لیں (لنک ہر بار بدلتا ہے)۔
- **🟡 ماڈل لوڈ ہو رہے ہیں:** پہلی بار 10 سے 15 منٹ لگتے ہیں، انتظار کریں۔
- **GPU کی حد پوری ہو گئی:** مفت Colab کی روزانہ حد ہوتی ہے۔ چند گھنٹے بعد دوبارہ کوشش کریں۔
- **لنک کسی کو نہ بھیجیں:** جس کے پاس لنک ہو وہ آپ کا GPU استعمال کر سکتا ہے۔